In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("joins").getOrCreate()

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [3]:
%%writefile patients.csv
patient_id,patient_name,city,age,gender,blood_group,insurance_status
101,Rahul Sharma,Hyderabad,35,Male,O+,Active
102,Priya Reddy,Bangalore,29,Female,A+,Active
103,Amit Kumar,Mumbai,42,Male,B+,Inactive
104,Sneha Patel,Chennai,31,Female,O+,Active
105,Farhan Ali,Delhi,55,Male,AB+,Active
106,Neha Singh,,38,Female,A+,Inactive
107,Arjun Verma,Pune,26,Male,B+,Active
108,Meera Nair,Kochi,48,Female,O-,Active
109,Kiran Rao,Hyderabad,33,Male,,Inactive
110,Nisha Reddy,Bangalore,41,Female,A+,Active

Overwriting patients.csv


In [4]:
%%writefile appointments.csv
appointment_id,patient_id,doctor_name,department,appointment_date,consultation_fee,status
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2500,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending

Overwriting appointments.csv


In [5]:
%%writefile patient_preferences.json
[
{
 "patient_id":101,
 "preferred_hospital":"Apollo",
 "contact":{
 "phone":"9876500011",
 "email":"rahul@gmail.com"
 }
},
{
 "patient_id":102,
 "preferred_hospital":"Yashoda",
 "contact":{
 "phone":null,
 "email":"priya@gmail.com"
 }
},
{
 "patient_id":103,
 "preferred_hospital":"Care",
 "contact":{
 "phone":"9876500013",
 "email":null
 }
},
{
 "patient_id":104,
 "preferred_hospital":null,
 "contact":{
 "phone":"9876500014",
 "email":"sneha@gmail.com"
 }
}
]


Overwriting patient_preferences.json


In [6]:
pdf=spark.read.csv("patients.csv",header=True,inferSchema=True)

In [7]:
adf=spark.read.csv("appointments.csv",header=True,inferSchema=True)

In [8]:
pdf.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)



In [9]:
adf.printSchema()

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: timestamp (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- status: string (nullable = true)



In [10]:
pdf.count()

10

In [11]:
adf.count()

10

In [12]:
pdf.show(5)

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows


In [13]:
adf.show(5)

+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|            2500|Completed|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+
only showing top 5 rows


In [14]:
pdf.select("city").distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [15]:
adf.select("department").distinct().show()

+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [16]:
pdf.write.mode("overwrite").parquet("patients_parquet")

In [17]:
p_parquet=spark.read.parquet("patients_parquet")

In [18]:
p_parquet.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [19]:
print("CSV Count:",pdf.count())

CSV Count: 10


In [20]:
print("Parquet Count:",p_parquet.count())

Parquet Count: 10


In [21]:
pdf.filter(col("city")=="Hyderabad").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [22]:
pdf.filter(col("gender")=="Female").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [23]:
pdf.filter(col("age")>40).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [24]:
adf.filter(col("status")=="Completed").show()

+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|2025-02-03 00:00:00|            2500|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|2025-02-05 00:00:00|            1500|Completed|
+---------

In [25]:
adf.filter(col("status")=="Pending").show()

+--------------+----------+-----------+-----------+-------------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+-------------------+----------------+-------+
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|            1000|Pending|
|          5010|       108|  Dr. Anita|Dermatology|2025-02-10 00:00:00|            NULL|Pending|
+--------------+----------+-----------+-----------+-------------------+----------------+-------+



In [26]:
adf.filter(col("consultation_fee")>1500).show()

+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|            2000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|2025-02-03 00:00:00|            2500|Completed|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+



In [27]:
pdf.filter(col("insurance_status")=="Active").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [28]:
pdf.filter(col("insurance_status")=="Inactive").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [29]:
pdf.filter(col("blood_group")=="O+").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [30]:
adf.filter(col("department")=="Cardiology").show()

+--------------+----------+-----------+----------+-------------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|   appointment_date|consultation_fee|   status|
+--------------+----------+-----------+----------+-------------------+----------------+---------+
|          5001|       101| Dr. Ramesh|Cardiology|2025-01-10 00:00:00|            1500|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|2025-01-20 00:00:00|            1500|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|2025-02-05 00:00:00|            1500|Completed|
+--------------+----------+-----------+----------+-------------------+----------------+---------+



In [31]:
pdf.filter(col("city").isNull()).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [32]:
pdf.filter(col("blood_group").isNull()).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [33]:
adf.filter(col("consultation_fee").isNull()).show()

+--------------+----------+-----------+-----------+-------------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+-------------------+----------------+-------+
|          5010|       108|  Dr. Anita|Dermatology|2025-02-10 00:00:00|            NULL|Pending|
+--------------+----------+-----------+-----------+-------------------+----------------+-------+



In [34]:
pdf.select([count(when(col(c).isNull(),c)).alias(c) for c in pdf.columns]).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+



In [35]:
adf.select([count(when(col(c).isNull(),c)).alias(c) for c in adf.columns]).show()

+--------------+----------+-----------+----------+----------------+----------------+------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|status|
+--------------+----------+-----------+----------+----------------+----------------+------+
|             0|         0|          0|         0|               0|               1|     0|
+--------------+----------+-----------+----------+----------------+----------------+------+



In [36]:
pdf = pdf.fillna({"city":"Unknown"})
pdf.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [37]:
pdf=pdf.fillna({"blood_group":"Not Available"})
pdf.show()

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|           A+|    

In [38]:
adf=adf.fillna({"consultation_fee":0})
adf.show()

+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|            2000|Completed|
|         

In [39]:
adf.dropna(subset=["consultation_fee"]).show()

+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|            2000|Completed|
|         

In [40]:
pdf=pdf.withColumn("data_quality_status",when(col("city").isNull()|col("blood_group").isNull(),"Incomplete").otherwise("Complete"))
pdf.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|           Complete|
|       108|  Meera 

In [41]:
pdf.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   10|
+-------------------+-----+



In [42]:
pdf.select(upper(col("patient_name")).alias("patient_name")).show()

+------------+
|patient_name|
+------------+
|RAHUL SHARMA|
| PRIYA REDDY|
|  AMIT KUMAR|
| SNEHA PATEL|
|  FARHAN ALI|
|  NEHA SINGH|
| ARJUN VERMA|
|  MEERA NAIR|
|   KIRAN RAO|
| NISHA REDDY|
+------------+



In [43]:
pdf.select(lower(col("patient_name")).alias("patient_name")).show()

+------------+
|patient_name|
+------------+
|rahul sharma|
| priya reddy|
|  amit kumar|
| sneha patel|
|  farhan ali|
|  neha singh|
| arjun verma|
|  meera nair|
|   kiran rao|
| nisha reddy|
+------------+



In [44]:
pdf.select("patient_name",length(col("patient_name")).alias("name_length")).show()

+------------+-----------+
|patient_name|name_length|
+------------+-----------+
|Rahul Sharma|         12|
| Priya Reddy|         11|
|  Amit Kumar|         10|
| Sneha Patel|         11|
|  Farhan Ali|         10|
|  Neha Singh|         10|
| Arjun Verma|         11|
|  Meera Nair|         10|
|   Kiran Rao|          9|
| Nisha Reddy|         11|
+------------+-----------+



In [45]:
pdf.select("patient_name",substring(col("patient_name"),1,3).alias("first_3_letters")).show()

+------------+---------------+
|patient_name|first_3_letters|
+------------+---------------+
|Rahul Sharma|            Rah|
| Priya Reddy|            Pri|
|  Amit Kumar|            Ami|
| Sneha Patel|            Sne|
|  Farhan Ali|            Far|
|  Neha Singh|            Neh|
| Arjun Verma|            Arj|
|  Meera Nair|            Mee|
|   Kiran Rao|            Kir|
| Nisha Reddy|            Nis|
+------------+---------------+



In [46]:
pdf=pdf.withColumn("age_group",when(col("age")<18,"Child").when(col("age")<60,"Adult").otherwise("Senior"))
pdf.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Adult|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|    Adult|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|    Adult|
|       107| Arjun Verma|   

In [47]:
pdf=pdf.withColumn("insurance_flag",when(col("insurance_status")=="Active","Y").otherwise("N"))
pdf.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             Y|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Adult|             Y|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|             N|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|             Y|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|    Adult|             Y|
|       106|  Ne

In [48]:
pdf=pdf.withColumn("senior_citizen",when(col("age")>=60,"Yes").otherwise("No"))
pdf.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             Y|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Adult|             Y|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|             N|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|             Y|            No|
|       105|  Farhan Ali|    Delhi

In [49]:
pdf.select(concat_ws(" - ",col("patient_name"),col("city")).alias("patient_city")).show()

+--------------------+
|        patient_city|
+--------------------+
|Rahul Sharma - Hy...|
|Priya Reddy - Ban...|
| Amit Kumar - Mumbai|
|Sneha Patel - Che...|
|  Farhan Ali - Delhi|
|Neha Singh - Unknown|
|  Arjun Verma - Pune|
|  Meera Nair - Kochi|
|Kiran Rao - Hyder...|
|Nisha Reddy - Ban...|
+--------------------+



In [50]:
pdf.select(trim(col("patient_name")).alias("patient_name")).show()

+------------+
|patient_name|
+------------+
|Rahul Sharma|
| Priya Reddy|
|  Amit Kumar|
| Sneha Patel|
|  Farhan Ali|
|  Neha Singh|
| Arjun Verma|
|  Meera Nair|
|   Kiran Rao|
| Nisha Reddy|
+------------+



In [51]:
pdf.select(upper(col("city")).alias("city")).show()

+---------+
|     city|
+---------+
|HYDERABAD|
|BANGALORE|
|   MUMBAI|
|  CHENNAI|
|    DELHI|
|  UNKNOWN|
|     PUNE|
|    KOCHI|
|HYDERABAD|
|BANGALORE|
+---------+



In [52]:
pdf.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    1|
|  Unknown|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [53]:
pdf.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [54]:
pdf.groupBy("blood_group").count().show()

+-------------+-----+
|  blood_group|count|
+-------------+-----+
|          AB+|    1|
|           O+|    2|
|           O-|    1|
|           B+|    2|
|           A+|    3|
|Not Available|    1|
+-------------+-----+



In [55]:
adf.groupBy("department").count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [56]:
pdf.groupBy("city").agg(avg("age").alias("avg_age")).show()

+---------+-------+
|     city|avg_age|
+---------+-------+
|Bangalore|   35.0|
|    Kochi|   48.0|
|  Chennai|   31.0|
|   Mumbai|   42.0|
|  Unknown|   38.0|
|     Pune|   26.0|
|    Delhi|   55.0|
|Hyderabad|   34.0|
+---------+-------+



In [57]:
pdf.groupBy("city").agg(max("age").alias("max_age")).show()

+---------+-------+
|     city|max_age|
+---------+-------+
|Bangalore|     41|
|    Kochi|     48|
|  Chennai|     31|
|   Mumbai|     42|
|  Unknown|     38|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     35|
+---------+-------+



In [58]:
pdf.groupBy("city").agg(min("age").alias("min_age")).show()

+---------+-------+
|     city|min_age|
+---------+-------+
|Bangalore|     29|
|    Kochi|     48|
|  Chennai|     31|
|   Mumbai|     42|
|  Unknown|     38|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     33|
+---------+-------+



In [59]:
adf.groupBy("department").agg(avg("consultation_fee").alias("avg_fee")).show()

+-----------+-----------------+
| department|          avg_fee|
+-----------+-----------------+
|  Neurology|           2000.0|
|Dermatology|666.6666666666666|
| Cardiology|           1500.0|
|Orthopedics|           2500.0|
+-----------+-----------------+



In [60]:
adf.groupBy("department").agg(sum("consultation_fee").alias("total_fee")).show()

+-----------+---------+
| department|total_fee|
+-----------+---------+
|  Neurology|     4000|
|Dermatology|     2000|
| Cardiology|     4500|
|Orthopedics|     5000|
+-----------+---------+



In [61]:
adf.groupBy("department").agg(sum("consultation_fee").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|Orthopedics|         5000|
+-----------+-------------+
only showing top 1 row


In [62]:
pdf.join(adf,"patient_id","inner").show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+-------------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+-------------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             Y|            No|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|       

In [63]:
pdf.join(adf,"patient_id","left").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+-------------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+-------------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             Y|            No|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adu

In [64]:
pdf.join(adf,"patient_id","right").show()

+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+-------------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+-------------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|           Complete|    Adult|             Y|            No|          5001| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|           Complete|    Adult|  

In [65]:
pdf.join(adf,"patient_id","full").show()

+----------+------------+---------+----+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+-------------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|   appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+-------------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|           O+|          Active|           Complete|    Adult|             Y|            No|          5001| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|           O+|          Active|           Complete|  

In [66]:
pdf.join(adf,"patient_id","left_anti").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|    Adult|             N|            No|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|           Complete|    Adult|             N|            No|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+



In [67]:
adf.join(pdf,"patient_id","left_anti").show()

+----------+--------------+-----------+----------+-------------------+----------------+---------+
|patient_id|appointment_id|doctor_name|department|   appointment_date|consultation_fee|   status|
+----------+--------------+-----------+----------+-------------------+----------------+---------+
|       120|          5009| Dr. Ramesh|Cardiology|2025-02-05 00:00:00|            1500|Completed|
+----------+--------------+-----------+----------+-------------------+----------------+---------+



In [68]:
adf.groupBy("patient_id").count().show()

+----------+-----+
|patient_id|count|
+----------+-----+
|       108|    1|
|       101|    2|
|       103|    1|
|       120|    1|
|       107|    1|
|       102|    1|
|       105|    1|
|       110|    1|
|       104|    1|
+----------+-----+



In [69]:
pdf.join(adf,"patient_id","inner").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).show()

+----------+------------+----------+
|patient_id|patient_name|total_fees|
+----------+------------+----------+
|       107| Arjun Verma|      2000|
|       108|  Meera Nair|         0|
|       110| Nisha Reddy|      2500|
|       105|  Farhan Ali|      1000|
|       101|Rahul Sharma|      2500|
|       104| Sneha Patel|      2500|
|       103|  Amit Kumar|      1500|
|       102| Priya Reddy|      2000|
+----------+------------+----------+



In [70]:
pdf.join(adf,"patient_id","inner").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).orderBy(desc("total_fees")).show(1)

+----------+------------+----------+
|patient_id|patient_name|total_fees|
+----------+------------+----------+
|       101|Rahul Sharma|      2500|
+----------+------------+----------+
only showing top 1 row


In [71]:
pdf.join(adf,"patient_id","inner").groupBy("patient_id","patient_name").count().show()

+----------+------------+-----+
|patient_id|patient_name|count|
+----------+------------+-----+
|       107| Arjun Verma|    1|
|       108|  Meera Nair|    1|
|       110| Nisha Reddy|    1|
|       105|  Farhan Ali|    1|
|       101|Rahul Sharma|    2|
|       104| Sneha Patel|    1|
|       103|  Amit Kumar|    1|
|       102| Priya Reddy|    1|
+----------+------------+-----+



In [72]:
fdf=pdf.join(adf,"patient_id","inner").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).withColumn("rank",rank().over(Window.orderBy(desc("total_fees"))))
fdf.show()

+----------+------------+----------+----+
|patient_id|patient_name|total_fees|rank|
+----------+------------+----------+----+
|       110| Nisha Reddy|      2500|   1|
|       101|Rahul Sharma|      2500|   1|
|       104| Sneha Patel|      2500|   1|
|       107| Arjun Verma|      2000|   4|
|       102| Priya Reddy|      2000|   4|
|       103|  Amit Kumar|      1500|   6|
|       105|  Farhan Ali|      1000|   7|
|       108|  Meera Nair|         0|   8|
+----------+------------+----------+----+



In [73]:
fdf=pdf.join(adf,"patient_id","inner").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).withColumn("dense_rank",dense_rank().over(Window.orderBy(desc("total_fees"))))
fdf.show()

+----------+------------+----------+----------+
|patient_id|patient_name|total_fees|dense_rank|
+----------+------------+----------+----------+
|       110| Nisha Reddy|      2500|         1|
|       101|Rahul Sharma|      2500|         1|
|       104| Sneha Patel|      2500|         1|
|       107| Arjun Verma|      2000|         2|
|       102| Priya Reddy|      2000|         2|
|       103|  Amit Kumar|      1500|         3|
|       105|  Farhan Ali|      1000|         4|
|       108|  Meera Nair|         0|         5|
+----------+------------+----------+----------+



In [74]:
fdf=pdf.join(adf,"patient_id","inner").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).withColumn("row_number",row_number().over(Window.orderBy(desc("total_fees"))))
fdf.show()

+----------+------------+----------+----------+
|patient_id|patient_name|total_fees|row_number|
+----------+------------+----------+----------+
|       110| Nisha Reddy|      2500|         1|
|       101|Rahul Sharma|      2500|         2|
|       104| Sneha Patel|      2500|         3|
|       107| Arjun Verma|      2000|         4|
|       102| Priya Reddy|      2000|         5|
|       103|  Amit Kumar|      1500|         6|
|       105|  Farhan Ali|      1000|         7|
|       108|  Meera Nair|         0|         8|
+----------+------------+----------+----------+



In [75]:
pdf.join(adf,"patient_id","inner").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).orderBy(desc("total_fees")).show(1)

+----------+------------+----------+
|patient_id|patient_name|total_fees|
+----------+------------+----------+
|       101|Rahul Sharma|      2500|
+----------+------------+----------+
only showing top 1 row


In [76]:
pdf.join(adf,"patient_id","inner").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).orderBy(desc("total_fees")).show(3)

+----------+------------+----------+
|patient_id|patient_name|total_fees|
+----------+------------+----------+
|       101|Rahul Sharma|      2500|
|       110| Nisha Reddy|      2500|
|       104| Sneha Patel|      2500|
+----------+------------+----------+
only showing top 3 rows


In [77]:
fdf=pdf.join(adf,"patient_id","inner").groupBy("city","patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).withColumn("rank",rank().over(Window.partitionBy("city").orderBy(desc("total_fees")))).filter(col("rank")==1)
fdf.show()

+---------+----------+------------+----------+----+
|     city|patient_id|patient_name|total_fees|rank|
+---------+----------+------------+----------+----+
|Bangalore|       110| Nisha Reddy|      2500|   1|
|  Chennai|       104| Sneha Patel|      2500|   1|
|    Delhi|       105|  Farhan Ali|      1000|   1|
|Hyderabad|       101|Rahul Sharma|      2500|   1|
|    Kochi|       108|  Meera Nair|         0|   1|
|   Mumbai|       103|  Amit Kumar|      1500|   1|
|     Pune|       107| Arjun Verma|      2000|   1|
+---------+----------+------------+----------+----+



In [78]:
fdf=pdf.join(adf,"patient_id","inner").groupBy("city","patient_id","patient_name").agg(sum("consultation_fee").alias("total_fees")).withColumn("rank",rank().over(Window.partitionBy("city").orderBy("total_fees"))).filter(col("rank")==1)
fdf.show()

+---------+----------+------------+----------+----+
|     city|patient_id|patient_name|total_fees|rank|
+---------+----------+------------+----------+----+
|Bangalore|       102| Priya Reddy|      2000|   1|
|  Chennai|       104| Sneha Patel|      2500|   1|
|    Delhi|       105|  Farhan Ali|      1000|   1|
|Hyderabad|       101|Rahul Sharma|      2500|   1|
|    Kochi|       108|  Meera Nair|         0|   1|
|   Mumbai|       103|  Amit Kumar|      1500|   1|
|     Pune|       107| Arjun Verma|      2000|   1|
+---------+----------+------------+----------+----+



In [79]:
fdf=adf.withColumn("running_total",sum("consultation_fee").over(Window.orderBy("appointment_id")))
fdf.show()

+--------------+----------+-----------+-----------+-------------------+----------------+---------+-------------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee|   status|running_total|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+-------------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|         1500|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|            2000|Completed|         3500|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|         4500|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|            1500|Cancelled|         6000|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|            2500|Completed|         8500|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|            1000|  Pendin

In [80]:
fdf=adf.withColumn("next_fee",lead("consultation_fee").over(Window.orderBy("appointment_id")))
fdf.show()

+--------------+----------+-----------+-----------+-------------------+----------------+---------+--------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee|   status|next_fee|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+--------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|    2000|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|            2000|Completed|    1000|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|    1500|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|            1500|Cancelled|    2500|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|            2500|Completed|    1000|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|            1000|  Pending|    2000|
|          5007|       107| 

In [81]:
fdf=adf.withColumn("previous_fee",lag("consultation_fee").over(Window.orderBy("appointment_id")))
fdf.show()

+--------------+----------+-----------+-----------+-------------------+----------------+---------+------------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consultation_fee|   status|previous_fee|
+--------------+----------+-----------+-----------+-------------------+----------------+---------+------------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            1500|Completed|        NULL|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|            2000|Completed|        1500|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|        2000|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|            1500|Cancelled|        1000|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|            2500|Completed|        1500|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|            1000|  Pending|      

In [82]:
%%writefile patient_preferences.json
[
{
 "patient_id":101,
 "preferred_hospital":"Apollo",
 "contact":{
  "phone":"9876500011",
  "email":"rahul@gmail.com"
 }
},
{
 "patient_id":102,
 "preferred_hospital":"Yashoda",
 "contact":{
  "phone":null,
  "email":"priya@gmail.com"
 }
},
{
 "patient_id":103,
 "preferred_hospital":"Care",
 "contact":{
  "phone":"9876500013",
  "email":null
 }
},
{
 "patient_id":104,
 "preferred_hospital":null,
 "contact":{
  "phone":"9876500014",
  "email":"sneha@gmail.com"
 }
}
]

Overwriting patient_preferences.json


In [83]:
jdf=spark.read.option("multiline","true").json("patient_preferences.json")

In [84]:
jdf.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [85]:
jdf.select("patient_id","contact.phone").show()

+----------+----------+
|patient_id|     phone|
+----------+----------+
|       101|9876500011|
|       102|      NULL|
|       103|9876500013|
|       104|9876500014|
+----------+----------+



In [86]:
jdf.select("patient_id","contact.email").show()

+----------+---------------+
|patient_id|          email|
+----------+---------------+
|       101|rahul@gmail.com|
|       102|priya@gmail.com|
|       103|           NULL|
|       104|sneha@gmail.com|
+----------+---------------+



In [87]:
jdf.filter(col("contact.phone").isNull()).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{priya@gmail.com,...|       102|           Yashoda|
+--------------------+----------+------------------+



In [88]:
jdf.filter(col("contact.email").isNull()).show()

+------------------+----------+------------------+
|           contact|patient_id|preferred_hospital|
+------------------+----------+------------------+
|{NULL, 9876500013}|       103|              Care|
+------------------+----------+------------------+



In [89]:
jdf.filter(col("preferred_hospital").isNull()).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{sneha@gmail.com,...|       104|              NULL|
+--------------------+----------+------------------+



In [90]:
jdf=jdf.withColumn("phone",coalesce(col("contact.phone"),lit("Not Available")))
jdf.show()

+--------------------+----------+------------------+-------------+
|             contact|patient_id|preferred_hospital|        phone|
+--------------------+----------+------------------+-------------+
|{rahul@gmail.com,...|       101|            Apollo|   9876500011|
|{priya@gmail.com,...|       102|           Yashoda|Not Available|
|  {NULL, 9876500013}|       103|              Care|   9876500013|
|{sneha@gmail.com,...|       104|              NULL|   9876500014|
+--------------------+----------+------------------+-------------+



In [91]:
jdf=jdf.withColumn("email",coalesce(col("contact.email"),lit("Not Available")))
jdf.show()

+--------------------+----------+------------------+-------------+---------------+
|             contact|patient_id|preferred_hospital|        phone|          email|
+--------------------+----------+------------------+-------------+---------------+
|{rahul@gmail.com,...|       101|            Apollo|   9876500011|rahul@gmail.com|
|{priya@gmail.com,...|       102|           Yashoda|Not Available|priya@gmail.com|
|  {NULL, 9876500013}|       103|              Care|   9876500013|  Not Available|
|{sneha@gmail.com,...|       104|              NULL|   9876500014|sneha@gmail.com|
+--------------------+----------+------------------+-------------+---------------+



In [92]:
pdf.join(jdf,"patient_id","left").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------------+------------------+-------------+---------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|             contact|preferred_hospital|        phone|          email|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------------+------------------+-------------+---------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             Y|            No|{rahul@gmail.com,...|            Apollo|   9876500011|rahul@gmail.com|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Adult|             Y|            No|{priya@gmail.com,...|          

In [93]:
pdf.createOrReplaceTempView("patients")

In [94]:
adf.createOrReplaceTempView("appointments")

In [95]:
spark.sql("select * from patients").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             Y|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Adult|             Y|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|             N|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|             Y|            No|
|       105|  Farhan Ali|    Delhi

In [96]:
spark.sql("select * from patients where city='Hyderabad'").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             Y|            No|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|           Complete|    Adult|             N|            No|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+



In [97]:
spark.sql("select city,count(*) as patient_count from patients group by city").show()

+---------+-------------+
|     city|patient_count|
+---------+-------------+
|Bangalore|            2|
|    Kochi|            1|
|  Chennai|            1|
|   Mumbai|            1|
|  Unknown|            1|
|     Pune|            1|
|    Delhi|            1|
|Hyderabad|            2|
+---------+-------------+



In [98]:
spark.sql("select department,count(*) as appointment_count from appointments group by department").show()

+-----------+-----------------+
| department|appointment_count|
+-----------+-----------------+
|  Neurology|                2|
|Dermatology|                3|
| Cardiology|                3|
|Orthopedics|                2|
+-----------+-----------------+



In [99]:
spark.sql("select department,avg(consultation_fee) as avg_fee from appointments group by department").show()

+-----------+-----------------+
| department|          avg_fee|
+-----------+-----------------+
|  Neurology|           2000.0|
|Dermatology|666.6666666666666|
| Cardiology|           1500.0|
|Orthopedics|           2500.0|
+-----------+-----------------+



In [100]:
spark.sql("select max(consultation_fee) as highest_fee from appointments").show()

+-----------+
|highest_fee|
+-----------+
|       2500|
+-----------+



In [101]:
spark.sql("select patient_id,count(*) as appointment_count from appointments group by patient_id").show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       108|                1|
|       101|                2|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [102]:
spark.sql("select patient_id,sum(consultation_fee) as total_spending from appointments group by patient_id order by total_spending desc limit 5").show()

+----------+--------------+
|patient_id|total_spending|
+----------+--------------+
|       101|          2500|
|       110|          2500|
|       104|          2500|
|       107|          2000|
|       102|          2000|
+----------+--------------+



In [103]:
pdf=spark.read.csv("patients.csv",header=True,inferSchema=True)
adf=spark.read.csv("appointments.csv",header=True,inferSchema=True)

In [104]:
jdf=spark.read.option("multiline","true").json("patient_preferences.json")

In [105]:
pdf=pdf.fillna({"city":"Unknown","blood_group":"Not Available"})
adf=adf.fillna({"consultation_fee":0})
jdf=jdf.withColumn("phone",coalesce(col("contact.phone"),lit("Not Available"))).withColumn("email",coalesce(col("contact.email"),lit("Not Available")))

In [106]:
fdf=pdf.join(adf,"patient_id","left").join(jdf.select("patient_id","preferred_hospital","phone","email"),"patient_id","left")
fdf.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----------------+---------+------------------+-------------+---------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|   appointment_date|consultation_fee|   status|preferred_hospital|        phone|          email|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----------------+---------+------------------+-------------+---------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|            Apollo|   9876500011|rahul@gmail.com|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|            

In [107]:
fdf=fdf.withColumn("age_group",when(col("age")<18,"Child").when(col("age")<60,"Adult").otherwise("Senior"))
fdf.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----------------+---------+------------------+-------------+---------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|   appointment_date|consultation_fee|   status|preferred_hospital|        phone|          email|age_group|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----------------+---------+------------------+-------------+---------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|            Apollo|   9876500011|rahul@gmail.com|    Adult|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Car

In [108]:
fdf=fdf.withColumn("revenue",col("consultation_fee"))
fdf.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----------------+---------+------------------+-------------+---------------+---------+-------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|   appointment_date|consultation_fee|   status|preferred_hospital|        phone|          email|age_group|revenue|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----------------+---------+------------------+-------------+---------------+---------+-------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|            1000|Completed|            Apollo|   9876500011|rahul@gmail.com|    Adult|   1000|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active

In [109]:
fdf.groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_spending")).show()

+----------+------------+--------------+
|patient_id|patient_name|total_spending|
+----------+------------+--------------+
|       107| Arjun Verma|          2000|
|       108|  Meera Nair|             0|
|       109|   Kiran Rao|          NULL|
|       110| Nisha Reddy|          2500|
|       105|  Farhan Ali|          1000|
|       101|Rahul Sharma|          2500|
|       104| Sneha Patel|          2500|
|       103|  Amit Kumar|          1500|
|       106|  Neha Singh|          NULL|
|       102| Priya Reddy|          2000|
+----------+------------+--------------+



In [110]:
fdf.groupBy("department").agg(sum("consultation_fee").alias("department_revenue")).show()

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|       NULL|              NULL|
|  Neurology|              4000|
|Dermatology|              2000|
| Cardiology|              3000|
|Orthopedics|              5000|
+-----------+------------------+



In [111]:
fdf.write.mode("overwrite").parquet("hospital_analytics_output")

In [112]:
from pyspark.sql.functions import countDistinct

In [113]:
fdf.groupBy("department").agg(
    countDistinct("patient_id").alias("total_patients"),
    count("appointment_id").alias("total_appointments"),
    sum("consultation_fee").alias("total_revenue"),
    avg("consultation_fee").alias("average_fee")
).show()

+-----------+--------------+------------------+-------------+-----------------+
| department|total_patients|total_appointments|total_revenue|      average_fee|
+-----------+--------------+------------------+-------------+-----------------+
|       NULL|             2|                 0|         NULL|             NULL|
|  Neurology|             2|                 2|         4000|           2000.0|
|Dermatology|             3|                 3|         2000|666.6666666666666|
| Cardiology|             2|                 2|         3000|           1500.0|
|Orthopedics|             2|                 2|         5000|           2500.0|
+-----------+--------------+------------------+-------------+-----------------+

